# Module 2 project-exercise demo — Segmentation metrics and a special-ID snapshot

**Implementation demo, not a completed full-scale project.** Two examples per language illustrate aggregation and denominator choice. `wordrun_proxy` counts letter/digit runs and punctuation, not WordPiece tokens; it discards spaces. A Chinese line is one letter run, not one linguistic word. These are toy segmenters, not a tokenizer ranking. Demo 2 reconstructs a dated Harmony snapshot rather than inspecting an installed library.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'


## 1. What does the denominator count?

We segment two short aligned phrases in each of four languages in three ways:
UTF-8 bytes, Unicode code points, and runs of letters/digits plus punctuation.
The last method discards spaces, so it is not a reversible tokenizer. We first
divide total units by total letter/digit runs, then by aligned sentence count.

Before running, consider why a Chinese line forms one run and how that affects
the left panel. Read each number together with its denominator. The panels use
separate color scales, so the same color need not mean the same number. Two phrases
illustrate metric mechanics; they cannot establish a language-wide efficiency claim.

In [ ]:
INK = '#2E3440'
MUTED = '#4C566A'
BORDER = '#D8DEE9'
def plot_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 12,
        "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
        "axes.titlesize": 17, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "axes.edgecolor": BORDER, "axes.linewidth": .7,
        "axes.facecolor": "white", "figure.facecolor": "white", "savefig.facecolor": "white",
        "grid.color": BORDER, "grid.linewidth": .7, "axes.axisbelow": True,
        "legend.fontsize": 11, "legend.frameon": False, "savefig.dpi": 150,
        "xtick.major.size": 0, "ytick.major.size": 0,
    })

plot_style()

import re

corpus = {
    "en": ["models learn from tokens", "a short test"],
    "ru": ["модели учатся на токенах", "короткий тест"],
    "es": ["modelos aprenden de tokens", "una prueba corta"],
    "zh": ["模型从词元学习", "简短测试"],
}

def word_runs(text):
    return re.findall(r"[^\W_]+", text, flags=re.UNICODE)

tokenizers = {
    "byte": lambda s: list(s.encode("utf-8")),
    "char": list,
    "wordrun_proxy": lambda s: re.findall(r"[^\W_]+|[^\w\s]", s, flags=re.UNICODE),
}

# The same aligned toy content; this is not a representative benchmark corpus.
fertility = np.zeros((len(corpus), len(tokenizers)))
units_per_sentence = np.zeros_like(fertility)
denominators = {}
for i, (lang, lines) in enumerate(corpus.items()):
    n_runs = sum(len(word_runs(line)) for line in lines)
    denominators[lang] = n_runs
    n_bytes = sum(len(line.encode("utf-8")) for line in lines)
    print(lang, "sentences / word runs / bytes:", len(lines), n_runs, n_bytes)
    for j, (name, encode) in enumerate(tokenizers.items()):
        n_units = sum(len(encode(line)) for line in lines)
        fertility[i, j] = n_units / n_runs
        units_per_sentence[i, j] = n_units / len(lines)
        print("  ", name, "units:", n_units, "units/sentence:", units_per_sentence[i, j])

tr = lambda ru, en: ru if DEMO_LANGUAGE == 'ru' else en
fig, axes = plt.subplots(1, 2, figsize=(11, 6), layout='constrained')
titles = [tr('Единиц на серию букв/цифр', 'Units per letter/digit run'),
          tr('Единиц на предложение', 'Units per aligned sentence')]
labels = [tr('Байты', 'Bytes'), tr('Кодовые\nточки', 'Code\npoints'),
          tr('Серии и\nзнаки', 'Runs and\npunctuation')]
for ax, values, title in zip(axes, [fertility, units_per_sentence], titles):
    im = ax.imshow(values, cmap='Blues', vmin=0, vmax=values.max(), aspect='auto')
    ax.set(xticks=range(len(tokenizers)), xticklabels=labels,
           yticks=range(len(corpus)), yticklabels=list(corpus))
    ax.grid(False)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, f'{values[i, j]:.1f}', ha='center', va='center',
                    color='white' if values[i, j] > values.max() * .65 else INK)
    ax.set_title(title, fontsize=14, pad=14)
    fig.colorbar(im, ax=ax, location='bottom', shrink=.9,
                 label=tr('Единиц; своя шкала у каждой панели', 'Units; each panel has its own scale'))
fig.suptitle(tr('Выбор знаменателя меняет сравнение', 'The denominator changes the comparison'),
             fontsize=17, fontweight='bold', color=INK)
plt.show()
assert denominators["zh"] == 2  # Two whole lines, NOT two linguistic words.
assert np.isfinite(fertility).all()
assert units_per_sentence.shape == (4, 3)
assert [len(word_runs(line)) for line in corpus["zh"]] == [1, 1]

## 2. Why can names outnumber IDs?

We reconstruct the special-name map from the tiktoken source pinned on 11 October
2026. Two names can share an ID: the Python dictionary counts names, whereas
`set(special_to_id.values())` counts distinct values. Before running, compare the
reserved range with `endofprompt`. Then explain the difference between 1091 names
and 1090 IDs. Ordinary tokens occupy a disjoint range, allowing us to add their
199998 IDs to the number of distinct special IDs.

In [ ]:
# Snapshot: 2026-10-11. Source: openai/tiktoken revision 4e71bbe0c078468e00fefbf94b39849389f346e5.
# Live library verification is an opt-in cell in the main Module-2 notebook.
mergeable_ids = 199_998
protocol_to_id = {
    "<|startoftext|>": 199_998,
    "<|endoftext|>": 199_999,
    "<|return|>": 200_002,
    "<|constrain|>": 200_003,
    "<|channel|>": 200_005,
    "<|start|>": 200_006,
    "<|end|>": 200_007,
    "<|message|>": 200_008,
    "<|call|>": 200_012,
    "<|endofprompt|>": 200_018,
}
reserved_to_id = {
    "<|reserved_200000|>": 200_000,
    "<|reserved_200001|>": 200_001,
    "<|reserved_200004|>": 200_004,
    "<|reserved_200009|>": 200_009,
    "<|reserved_200010|>": 200_010,
    "<|reserved_200011|>": 200_011,
    **{f"<|reserved_{i}|>": i for i in range(200_013, 201_088)},
}
special_to_id = protocol_to_id | reserved_to_id
special_ids = set(special_to_id.values())
aliases = {}
for name, token_id in special_to_id.items():
    aliases.setdefault(token_id, []).append(name)
aliases = {token_id: names for token_id, names in aliases.items() if len(names) > 1}

print("mergeable ids:", mergeable_ids)
print("special names / unique ids:", len(special_to_id), len(special_ids))
print("aliases:", aliases)
print("total unique vocabulary ids:", mergeable_ids + len(special_ids))
assert len(protocol_to_id) == 10
assert len(reserved_to_id) == 1_081
assert len(special_to_id) == 1_091
assert len(special_ids) == 1_090
assert aliases == {200_018: ["<|endofprompt|>", "<|reserved_200018|>"]}
assert mergeable_ids + len(special_ids) == 201_088

## Scaling to the full project

Replace the teaching corpus with aligned FLORES rows for ten languages and the simplified tokenizers with pinned real artifacts. Record tokens per aligned sentence, bytes, and artifact revisions. For fertility, define appropriate word segmentation for each language; do not label Chinese regex runs as words.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.